In [ ]:
import sys
print(sys.executable)

# Phase 5 Slice 5B Safe Read-Only Notebook Demo

This notebook is a local POC client only. It signs the user in with Microsoft Entra device code, obtains only Token A, sends one read-only request through the unified Solo API listener, and never accesses Token B or Token C. It does not manage Docker and is not a production-readiness demonstration.

# Phase 5 Slice 5B Safe Read-Only Notebook Demo

This notebook is a local POC client only. It signs the user in with Microsoft Entra device code, obtains only Token A, sends one read-only request through the unified Solo API listener, and never accesses Token B or Token C. It does not manage Docker and is not a production-readiness demonstration.

In [ ]:
import json
import os
import re
import sys
import uuid
import urllib.error
import urllib.request
import msal
from dotenv import load_dotenv
from pathlib import Path
from urllib.parse import urlparse

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "agents").exists() and (REPO_ROOT.parent / "agents").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env", override=False)


class NotebookValidationError(RuntimeError):
    pass


def _sanitize_business_text(text: str) -> str:
    sanitized = text.strip()
    sanitized = re.sub(r"\b(?:acct|user)-\d+\b", "[redacted]", sanitized)
    sanitized = re.sub(r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b", "[redacted]", sanitized)
    return sanitized


def _load_base_url() -> str:
    base_url = os.getenv("SOLO_API_BASE_URL", "http://127.0.0.1:18080").strip()
    parsed = urlparse(base_url)
    if parsed.scheme not in {"http", "https"} or parsed.hostname not in {"127.0.0.1", "localhost", "::1"}:
        raise NotebookValidationError("Configuration failed: Solo API base URL must target a loopback endpoint.")
    return base_url.rstrip("/")


def _load_token_a() -> str:
    tenant_id = os.getenv("ENTRA_TENANT_ID", "").strip()
    client_id = os.getenv("ENTRA_NOTEBOOK_CLIENT_ID", "").strip()
    scope = os.getenv("ENTRA_AGENT_A_SCOPE", "").strip()
    if not tenant_id or not client_id or not scope:
        raise NotebookValidationError("Configuration failed: notebook Entra settings are incomplete.")
    authority = f"https://login.microsoftonline.com/{tenant_id}"
    client = msal.PublicClientApplication(client_id=client_id, authority=authority)
    flow = client.initiate_device_flow(scopes=[scope])
    message = flow.get("message") if isinstance(flow, dict) else None
    if not isinstance(message, str) or not message.strip():
        raise NotebookValidationError("Device-code sign-in could not be started.")
    print(message)
    result = client.acquire_token_by_device_flow(flow)
    token = result.get("access_token") if isinstance(result, dict) else None
    if not isinstance(token, str) or not token.strip():
        raise NotebookValidationError("Device-code sign-in did not return Token A.")
    return token.strip()


def _http_json(url: str, *, method: str, headers: dict[str, str], payload: object | None, error_prefix: str):
    request = urllib.request.Request(url, method=method, headers=headers)
    if payload is not None:
        request.add_header("Content-Type", "application/json")
        request.data = json.dumps(payload).encode("utf-8")
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            body = response.read().decode("utf-8")
            parsed_body = json.loads(body) if body else None
            return response.status, parsed_body, dict(response.headers)
    except urllib.error.HTTPError as exc:
        raise NotebookValidationError(f"{error_prefix} failed: HTTP {exc.code}.") from exc
    except urllib.error.URLError as exc:
        raise NotebookValidationError(f"{error_prefix} failed: network unavailable.") from exc
    except json.JSONDecodeError as exc:
        raise NotebookValidationError(f"{error_prefix} failed: response was not valid JSON.") from exc


def _http_status(url: str, *, headers: dict[str, str], error_prefix: str) -> int:
    request = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(request, timeout=30) as response:
            response.read()
            return response.status
    except urllib.error.HTTPError as exc:
        return exc.code
    except urllib.error.URLError as exc:
        raise NotebookValidationError(f"{error_prefix} failed: network unavailable.") from exc


FLOW_ID = f"slice-5b-{uuid.uuid4().hex[:12]}"
SOLO_API_BASE_URL = _load_base_url()
TOKEN_A = _load_token_a()
APP_NAME = "agent_a"
USER_ID = "user-12345"

In [ ]:
readiness_status = _http_status(
    f"{SOLO_API_BASE_URL}/health",
    headers={"Authorization": f"Bearer {TOKEN_A}"},
    error_prefix="Readiness check",
)
READINESS = {
    "endpoint": SOLO_API_BASE_URL,
    "readiness_status": readiness_status == 200,
    "flow_id": FLOW_ID,
    "status_category": "ready" if readiness_status == 200 else f"http-{readiness_status}",
}
print(json.dumps(READINESS, indent=2))

In [ ]:
session_status, session_body, _ = _http_json(
    f"{SOLO_API_BASE_URL}/apps/{APP_NAME}/users/{USER_ID}/sessions",
    method="POST",
    headers={
        "Authorization": f"Bearer {TOKEN_A}",
        "Content-Type": "application/json",
    },
    payload={},
    error_prefix="Session creation",
)
if session_status != 200 or not isinstance(session_body, dict):
    raise NotebookValidationError("Session creation failed: unexpected response contract.")
SESSION_ID = session_body.get("id")
if not isinstance(SESSION_ID, str) or not SESSION_ID.strip():
    raise NotebookValidationError("Session creation failed: session identifier was not returned.")
SESSION_CREATED = True

In [ ]:
PROMPT_TEXT = (
    "What is the balance for synthetic account acct-1001? Use only the approved read-only banking path and keep the answer brief."
)
RUN_PAYLOAD = {
    "app_name": APP_NAME,
    "user_id": USER_ID,
    "session_id": SESSION_ID,
    "new_message": {
        "role": "user",
        "parts": [{"text": PROMPT_TEXT}],
    },
}
RUN_STATUS, RUN_EVENTS, RUN_HEADERS = _http_json(
    f"{SOLO_API_BASE_URL}/run",
    method="POST",
    headers={
        "Authorization": f"Bearer {TOKEN_A}",
        "Content-Type": "application/json",
        "X-Flow-ID": FLOW_ID,
    },
    payload=RUN_PAYLOAD,
    error_prefix="Read-only run",
)
if RUN_STATUS != 200 or not isinstance(RUN_EVENTS, list):
    raise NotebookValidationError("Read-only run failed: unexpected response contract.")
RESPONSE_FLOW_ID = next((value for key, value in RUN_HEADERS.items() if key.lower() == "x-flow-id"), None)
if RESPONSE_FLOW_ID != FLOW_ID:
    raise NotebookValidationError("Read-only run failed: flow ID was not echoed by the API boundary.")

In [ ]:
def _iter_parts(run_events: object):
    if not isinstance(run_events, list):
        raise NotebookValidationError("Read-only run failed: response schema did not match the proven contract.")
    for event in run_events:
        if not isinstance(event, dict):
            raise NotebookValidationError("Read-only run failed: response schema did not match the proven contract.")
        content = event.get("content")
        parts = event.get("parts")
        if isinstance(content, dict):
            parts = content.get("parts")
        if parts is None:
            continue
        if not isinstance(parts, list):
            raise NotebookValidationError("Read-only run failed: response schema did not match the proven contract.")
        yield event, parts


def _tool_payload(part: object) -> dict[str, object] | None:
    if not isinstance(part, dict):
        return None
    for key in (
        "functionCall",
        "functionResponse",
        "function_call",
        "function_response",
        "toolCall",
        "toolResponse",
        "tool_call",
        "tool_response",
    ):
        value = part.get(key)
        if isinstance(value, dict):
            return value
    return None


def _tool_name(payload: dict[str, object] | None) -> str:
    if not isinstance(payload, dict):
        return ""
    for key in ("name", "tool_name", "toolName"):
        value = payload.get(key)
        if isinstance(value, str):
            return value
    return ""


def _extract_final_agent_a_text(run_events: object) -> str:
    if not isinstance(run_events, list):
        raise NotebookValidationError("Read-only run failed: response schema did not match the proven contract.")
    for event in reversed(run_events):
        if not isinstance(event, dict):
            raise NotebookValidationError("Read-only run failed: response schema did not match the proven contract.")
        if str(event.get("author") or "").lower() != "agent_a":
            continue
        content = event.get("content")
        parts = event.get("parts")
        if isinstance(content, dict):
            parts = content.get("parts")
        if parts is None:
            continue
        if not isinstance(parts, list):
            raise NotebookValidationError("Read-only run failed: response schema did not match the proven contract.")
        texts = []
        for part in parts:
            if not isinstance(part, dict):
                raise NotebookValidationError("Read-only run failed: response schema did not match the proven contract.")
            if bool(part.get("thought")):
                continue
            if _tool_payload(part) is not None:
                continue
            part_type = str(part.get("type") or "").lower()
            if part_type and "text" not in part_type:
                continue
            text = part.get("text")
            if isinstance(text, str) and text.strip():
                texts.append(text.strip())
        if texts:
            return _sanitize_business_text("\n".join(texts))
    return ""


def _extract_text_from_delegate_response(value: object) -> str:
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, list):
        for item in value:
            text = _extract_text_from_delegate_response(item)
            if text:
                return text
        return ""
    if isinstance(value, dict):
        for key in ("result", "output", "text", "content"):
            if key not in value:
                continue
            text = _extract_text_from_delegate_response(value[key])
            if text:
                return text
    return ""


def _extract_delegate_to_agent_b_response(run_events: object) -> str:
    event_parts = list(_iter_parts(run_events))
    for _, parts in reversed(event_parts):
        for part in reversed(parts):
            if not isinstance(part, dict):
                raise NotebookValidationError("Read-only run failed: response schema did not match the proven contract.")
            for key in ("functionResponse", "function_response", "toolResponse", "tool_response"):
                payload = part.get(key)
                if not isinstance(payload, dict) or _tool_name(payload) != "delegate_to_agent_b":
                    continue
                text = _extract_text_from_delegate_response(payload.get("response"))
                if text:
                    return _sanitize_business_text(text)
    raise NotebookValidationError("Read-only run failed: neither final Agent A text nor the proven delegation response was available.")


def _contains_string(value: object, predicate) -> bool:
    if isinstance(value, str):
        return bool(predicate(value))
    if isinstance(value, list):
        return any(_contains_string(item, predicate) for item in value)
    if isinstance(value, dict):
        return any(_contains_string(item, predicate) for item in value.values())
    return False


tool_call_names = []
tool_response_names = []
for _, parts in _iter_parts(RUN_EVENTS):
    for part in parts:
        payload = _tool_payload(part)
        name = _tool_name(payload)
        if not name:
            continue
        if any(key in part for key in ("functionCall", "function_call", "toolCall", "tool_call")):
            tool_call_names.append(name)
        if any(key in part for key in ("functionResponse", "function_response", "toolResponse", "tool_response")):
            tool_response_names.append(name)

all_tool_names = {name.lower() for name in tool_call_names + tool_response_names}
allowed_tool_names = {"delegate_to_agent_b", "get_account_summary"}
unexpected_tool_names = sorted(all_tool_names - allowed_tool_names)
if unexpected_tool_names:
    raise NotebookValidationError("Read-only run failed: an unexpected tool appeared in the client-visible response.")

if _contains_string(RUN_EVENTS, lambda text: any(marker in text.lower() for marker in ("mutation", "write confirmation", "idempotency"))):
    raise NotebookValidationError("Read-only run failed: a write or mutation signal appeared in the client-visible response.")

FINAL_RESULT = _extract_final_agent_a_text(RUN_EVENTS)
if not FINAL_RESULT:
    FINAL_RESULT = _extract_delegate_to_agent_b_response(RUN_EVENTS)
normalized_result = FINAL_RESULT.replace(",", "")
if "12345.67" not in normalized_result:
    raise NotebookValidationError("Read-only run failed: the expected read-only balance signal was absent.")

SUMMARY = {
    "final_business_result": FINAL_RESULT,
    "flow_id": FLOW_ID,
    "session_run_success": True,
    "delegated_to_agent_b": "delegate_to_agent_b" in all_tool_names,
    "a2a_response_received": "delegate_to_agent_b" in {name.lower() for name in tool_response_names},
    "mcp_tool_call_client_visible": "get_account_summary" in all_tool_names,
    "mcp_tool_response_client_visible": "get_account_summary" in {name.lower() for name in tool_response_names},
    "mcp_lifecycle_visibility": "Agent B internal MCP events are verified server-side and are not required in the Agent A /run response",
    "mcp_failure_detected": _contains_string(RUN_EVENTS, lambda text: "mcp_failure" in text.lower()),
}
if SUMMARY["mcp_failure_detected"]:
    raise NotebookValidationError("Read-only run failed: MCP failure was detected in the client-visible response.")
if not SUMMARY["delegated_to_agent_b"] or not SUMMARY["a2a_response_received"]:
    raise NotebookValidationError("Read-only run failed: the client-visible response did not prove the completed Agent A delegation path.")
print(json.dumps(SUMMARY, indent=2))

## Limitations

This notebook validates a synthetic local POC path only. It does not prove production readiness, does not validate Phase 6, keeps FastMCP and MACAW as planned future work, does not own Token B or Token C, and does not manage service lifecycle.